In [1]:
import sys

print("Python executable:", sys.executable)

# Safety check – prevents silent failures
assert "azureml_py38" in sys.executable, "Wrong kernel/environment selected!"

Python executable: /anaconda/envs/azureml_py38/bin/python


In [2]:
# Basic sanity check
import sys
import torch

print("Python version:", sys.version)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# We explicitly want CPU for dynamic quantization
assert not torch.cuda.is_available(), "This notebook is designed for CPU quantization"

Python version: 3.10.11 (main, May 16 2023, 00:28:57) [GCC 11.2.0]
Torch version: 2.7.1+cu126
CUDA available: False


In [3]:
import os
import time
import psutil
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# We want CPU for dynamic INT8
assert torch.cuda.is_available() is False


Torch version: 2.7.1+cu126
CUDA available: False


In [5]:
import urllib3
import botocore
import transformers
import accelerate

print("urllib3:", urllib3.__version__)
print("botocore:", botocore.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)


urllib3: 1.26.20
botocore: 1.34.162
transformers: 4.48.0
accelerate: 0.34.2


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
MODEL_NAME = "distilgpt2"

print("Loading FP32 model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model_fp32.eval()

Loading FP32 model...


2026-01-14 05:27:07.141089: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768368428.456443    8122 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768368428.840726    8122 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768368434.707106    8122 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768368434.707173    8122 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768368434.707193    8122 computation_placer.cc:177] computation placer alr

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
def measure_latency(model, tokenizer, prompt, runs=5):
    inputs = tokenizer(prompt, return_tensors="pt")

    # Warm-up
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=20)

    times = []
    for _ in range(runs):
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=20)
        times.append(time.time() - start)

    return sum(times) / len(times)


def get_process_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)


In [8]:
prompt = "Explain quantization in simple terms."

fp32_latency = measure_latency(model_fp32, tokenizer, prompt)
fp32_memory = get_process_memory_mb()

print(f"FP32 Avg Latency: {fp32_latency:.3f} sec")
print(f"FP32 Process Memory: {fp32_memory:.1f} MB")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


FP32 Avg Latency: 0.493 sec
FP32 Process Memory: 1669.9 MB


In [9]:
print("Applying dynamic INT8 quantization...")

model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},   # Quantize only Linear layers
    dtype=torch.qint8
)

model_int8.eval()


Applying dynamic INT8 quantization...


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): DynamicQuantizedLinear(in_features=768, out_features=50257, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)

In [10]:
int8_latency = measure_latency(model_int8, tokenizer, prompt)
int8_memory = get_process_memory_mb()

print(f"INT8 Avg Latency: {int8_latency:.3f} sec")
print(f"INT8 Process Memory: {int8_memory:.1f} MB")

print("\nImprovement Summary:")
print(f"Latency improvement: {(fp32_latency - int8_latency) / fp32_latency * 100:.1f}%")
print(f"Runtime memory reduction: {fp32_memory - int8_memory:.1f} MB")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


INT8 Avg Latency: 0.296 sec
INT8 Process Memory: 2035.8 MB

Improvement Summary:
Latency improvement: 40.1%
Runtime memory reduction: -365.9 MB


In [11]:
BASE_DIR = "quantized_model"
MODEL_DIR = os.path.join(BASE_DIR, "model")
TOKENIZER_DIR = os.path.join(BASE_DIR, "tokenizer")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print("Created folder structure:")
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")


Created folder structure:
quantized_model/
  model/
  tokenizer/


In [12]:
# Save INT8 model weights
torch.save(
    model_int8.state_dict(),
    os.path.join(MODEL_DIR, "pytorch_model.bin")
)

# Save tokenizer
tokenizer.save_pretrained(TOKENIZER_DIR)

print("Model and tokenizer saved.")


Model and tokenizer saved.


In [13]:
inference_code = """
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def init():
    global model, tokenizer
    tokenizer = AutoTokenizer.from_pretrained("tokenizer")
    model = AutoModelForCausalLM.from_pretrained(
        "model",
        state_dict=torch.load("model/pytorch_model.bin", map_location="cpu")
    )
    model.eval()

def run(data):
    prompt = data.get("prompt", "")
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)

    return {
        "response": tokenizer.decode(outputs[0], skip_special_tokens=True)
    }
"""

with open(os.path.join(BASE_DIR, "inference.py"), "w") as f:
    f.write(inference_code)

print("inference.py created.")


inference.py created.


In [14]:
requirements = """
torch
transformers
accelerate
"""

with open(os.path.join(BASE_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

print("requirements.txt created.")


requirements.txt created.


In [15]:
print("Final artifact structure:\n")

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")


Final artifact structure:

quantized_model/
  .amlignore
  .amlignore.amltmp
  inference.py
  requirements.txt
  model/
    .amlignore
    .amlignore.amltmp
    pytorch_model.bin
  tokenizer/
    .amlignore
    .amlignore.amltmp
    merges.txt
    special_tokens_map.json
    tokenizer.json
    tokenizer_config.json
    vocab.json


In [16]:
import os

!ls quantized_model

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
!mkdir -p outputs/quantized_model
!cp -r quantized_model/* outputs/quantized_model/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
!ls quantized_model

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
!zip -r quantized_model_int8.zip quantized_model

  adding: quantized_model/ (stored 0%)
  adding: quantized_model/.amlignore (deflated 32%)
  adding: quantized_model/.amlignore.amltmp (deflated 32%)
  adding: quantized_model/inference.py (deflated 48%)
  adding: quantized_model/model/ (stored 0%)
  adding: quantized_model/model/.amlignore (deflated 32%)
  adding: quantized_model/model/.amlignore.amltmp (deflated 32%)
  adding: quantized_model/model/pytorch_model.bin (deflated 9%)
  adding: quantized_model/requirements.txt (stored 0%)
  adding: quantized_model/tokenizer/ (stored 0%)
  adding: quantized_model/tokenizer/.amlignore (deflated 32%)
  adding: quantized_model/tokenizer/.amlignore.amltmp (deflated 32%)
  adding: quantized_model/tokenizer/merges.txt (deflated 53%)
  adding: quantized_model/tokenizer/special_tokens_map.json (deflated 52%)
  adding: quantized_model/tokenizer/tokenizer.json (deflated 82%)
  adding: quantized_model/tokenizer/tokenizer_config.json (deflated 52%)
  adding: quantized_model/tokenizer/vocab.json (defla